# 05 — Build Vector Search From Scratch

We have embeddings sitting in a JSON file from notebook 04. A pile of numbers isn't useful on its own — we need to *search* it: given a question's embedding, find which stories are closest in meaning.

Before reaching for any vector database library, let's build one ourselves. It's about 30 lines. This is the most important notebook in the series — everything a production vector database does is an optimized version of exactly what we're about to write.


In [ ]:
import os
import json

DATA_DIR = os.path.join("..", "data")
embeddings_path = os.path.join(DATA_DIR, "embeddings.json")

with open(embeddings_path) as f:
    embedded_docs = json.load(f)

print(f"Loaded {len(embedded_docs)} documents")
print("First doc:", embedded_docs[0]["filename"], "-- embedding starts with", embedded_docs[0]["embedding"][:5])


## The vector store, one method at a time

A vector store really only needs to do two things: **hold documents with their embeddings**, and **rank them by similarity to a query embedding**.


In [ ]:
import math


class SimpleVectorStore:
    def __init__(self):
        self.documents = []  # list of {"content", "metadata", "embedding"}

    def add_document(self, content: str, embedding: list[float], metadata: dict | None = None):
        """Store one document alongside its embedding."""
        self.documents.append({"content": content, "metadata": metadata or {}, "embedding": embedding})

    def _cosine_similarity(self, a: list[float], b: list[float]) -> float:
        dot = sum(x * y for x, y in zip(a, b))
        mag_a = math.sqrt(sum(x**2 for x in a))
        mag_b = math.sqrt(sum(x**2 for x in b))
        return dot / (mag_a * mag_b)

    def similarity_search(self, query_embedding: list[float], top_k: int = 3) -> list[dict]:
        """Return the top_k documents ranked by similarity to the query."""
        scored = [
            {"document": doc, "score": self._cosine_similarity(query_embedding, doc["embedding"])}
            for doc in self.documents
        ]
        scored.sort(key=lambda x: x["score"], reverse=True)
        return scored[:top_k]


That's the whole class. `add_document` stores things; `similarity_search` compares the query against every stored document and returns the closest ones. Let's load our 5 stories into it.


In [ ]:
store = SimpleVectorStore()
for doc in embedded_docs:
    store.add_document(doc["content"], doc["embedding"], metadata={"filename": doc["filename"]})

print(f"Store holds {len(store.documents)} documents")


## Does it actually work?

Query: "space travel through a wormhole" — should surface Interstellar, without us ever typing that title.


In [ ]:
from sentence_transformers import SentenceTransformer

embedder = SentenceTransformer("all-MiniLM-L6-v2")

query = "space travel through a wormhole"
query_embedding = embedder.encode([query])[0].tolist()

results = store.similarity_search(query_embedding, top_k=3)
for r in results:
    print(f"({r['score']:.2f}) {r['document']['metadata']['filename']}")


Interstellar should be sitting at rank 1 — the model understood "space travel through a wormhole" is close to the actual plot, even though we never typed the word "Interstellar" anywhere in the query.


## Testing across all 5 stories

One query per story, checking that each surfaces its own story as the top result.


In [ ]:
test_queries = {
    "space travel through a wormhole": "interstellar.txt",
    "a boy discovers he is a wizard": "harry_potter_sorcerers_stone.txt",
    "a warrior reclaims his father's throne": "baahubali.txt",
    "a portal to another world inside a shop": "portal_bookshop.txt",
    "a masked vigilante protects a corrupt city": "dark_knight.txt",
}

for q, expected in test_queries.items():
    query_embedding = embedder.encode([q])[0].tolist()
    top_result = store.similarity_search(query_embedding, top_k=1)[0]
    got = top_result["document"]["metadata"]["filename"]
    status = "OK" if got == expected else "MISS"
    print(f"[{status}] '{q}' -> {got} (score {top_result['score']:.2f})")


Our 30-line vector store works correctly, on all 5 stories. So why does anyone bother with FAISS or Chroma at all, if 30 lines already does the job?

Because "works on 5 stories" is hiding a few cracks that only show up once StoryVerse AI has a real catalog:

- **O(n) search** — every query compares against *every* document. Fine for 5. For 5 million, that's minutes per query, not milliseconds.
- **Everything lives in a Python list, in memory** — restart the notebook, and the whole index is gone.
- **No persistence** — nothing touches disk unless we write that ourselves.
- **One embedding at a time** — no batching, no optimization.

This is precisely the gap FAISS was built to close.


## What FAISS actually is

**FAISS** (Facebook AI Similarity Search) is a library purpose-built for fast similarity search, including over millions or billions of vectors. It uses **approximate nearest neighbor (ANN)** search — trading a small amount of exactness for a large amount of speed — via index structures like `Flat` (exact, what we'll use), `IVF`, and `HNSW` (both approximate, faster at huge scale).

The core idea: FAISS does the *same thing* our `SimpleVectorStore` does, just engineered to be dramatically faster once you're past a few thousand documents. Under 10,000 documents, honestly, our pure-Python version is fine — FAISS starts mattering at scale.


In [ ]:
import faiss
import numpy as np


class FAISSVectorStore:
    def __init__(self, dimension: int = 384):
        # IndexFlatIP = exact search via inner product. On L2-normalized
        # vectors, inner product is mathematically equivalent to cosine similarity.
        self.index = faiss.IndexFlatIP(dimension)
        self.documents = []

    def add_document(self, content: str, embedding: list[float], metadata: dict | None = None):
        vec = np.array([embedding], dtype=np.float32)
        faiss.normalize_L2(vec)
        self.index.add(vec)
        self.documents.append({"content": content, "metadata": metadata or {}})

    def similarity_search(self, query_embedding: list[float], top_k: int = 3) -> list[dict]:
        query_vec = np.array([query_embedding], dtype=np.float32)
        faiss.normalize_L2(query_vec)
        scores, indices = self.index.search(query_vec, top_k)
        return [
            {"document": self.documents[i], "score": float(scores[0][j])}
            for j, i in enumerate(indices[0])
        ]


faiss_store = FAISSVectorStore(dimension=384)
for doc in embedded_docs:
    faiss_store.add_document(doc["content"], doc["embedding"], metadata={"filename": doc["filename"]})

print(f"FAISS index holds {faiss_store.index.ntotal} vectors")


Every line has a reason: `normalize_L2` scales vectors to length 1, so inner product behaves exactly like cosine similarity. `IndexFlatIP` means "no shortcuts, compare against everything, using inner product" — the exact FAISS equivalent of our own `_cosine_similarity` loop.


Let's put that claim to the test — same 3 queries, both stores, and see if they even agree with each other before we talk about speed.


In [ ]:
for q in list(test_queries.keys())[:3]:
    query_embedding = embedder.encode([q])[0].tolist()
    simple_top = store.similarity_search(query_embedding, top_k=1)[0]
    faiss_top = faiss_store.similarity_search(query_embedding, top_k=1)[0]
    print(f"Query: {q}")
    print(f"  SimpleVectorStore -> {simple_top['document']['metadata']['filename']} ({simple_top['score']:.3f})")
    print(f"  FAISSVectorStore  -> {faiss_top['document']['metadata']['filename']} ({faiss_top['score']:.3f})")


On 5 documents, both stores are effectively instant and agree with each other. Now let's simulate what happens at real scale.


In [ ]:
import time

NUM_FAKE = 10_000
rng = np.random.default_rng(42)
fake_embeddings = rng.random((NUM_FAKE, 384)).astype(np.float32)
query_vec = fake_embeddings[0]

# Pure Python store at scale
big_simple_store = SimpleVectorStore()
for i, emb in enumerate(fake_embeddings):
    big_simple_store.add_document(f"doc_{i}", emb.tolist())

start = time.time()
big_simple_store.similarity_search(query_vec.tolist(), top_k=5)
simple_time = time.time() - start

# FAISS store at scale
big_faiss_store = FAISSVectorStore(dimension=384)
for i, emb in enumerate(fake_embeddings):
    big_faiss_store.add_document(f"doc_{i}", emb.tolist())

start = time.time()
big_faiss_store.similarity_search(query_vec.tolist(), top_k=5)
faiss_time = time.time() - start

print(f"{NUM_FAKE:,} documents:")
print(f"  Pure Python search: {simple_time*1000:.1f} ms")
print(f"  FAISS search:       {faiss_time*1000:.1f} ms")


At 10,000 vectors the gap should already be visible; keep scaling `NUM_FAKE` up and it only widens. Same math, engineered implementation.


## One more layer up: Chroma

FAISS just solved our speed problem. But look back at `FAISSVectorStore.__init__` — we're still keeping a separate `self.documents` list ourselves, by hand, alongside the index. FAISS only ever agreed to store vectors; documents, metadata, and persistence were still our job. **Chroma** is a vector database that bundles all of that — embeddings, documents, metadata, persistence — behind one friendlier API.


In [ ]:
import chromadb

chroma_client = chromadb.Client()
collection = chroma_client.create_collection("storyverse")

collection.add(
    documents=[d["content"] for d in embedded_docs],
    embeddings=[d["embedding"] for d in embedded_docs],
    metadatas=[{"filename": d["filename"]} for d in embedded_docs],
    ids=[d["filename"] for d in embedded_docs],
)

query_embedding = embedder.encode(["space travel through a wormhole"])[0].tolist()
chroma_results = collection.query(query_embeddings=[query_embedding], n_results=3)

for filename, distance in zip(chroma_results["ids"][0], chroma_results["distances"][0]):
    print(f"({distance:.3f}) {filename}")


Chroma is, functionally, our `FAISSVectorStore` class with persistence, metadata filtering, and a nicer API bolted on. Nothing about the underlying idea changed.


## The progression, laid out

```
Python list -> SimpleVectorStore (ours) -> FAISS -> Chroma -> Pinecone / Qdrant

Increasing:  scale, features, persistence, cost
Decreasing:  simplicity, control, transparency
```

Start with what you understand. Reach for the next tool only once you've actually hit the wall the current one has.


## Saving our FAISS index for later

Notebook 07 will reuse this index instead of rebuilding it from scratch.


In [ ]:
faiss_index_path = os.path.join(DATA_DIR, "storyverse.faiss")
faiss.write_index(faiss_store.index, faiss_index_path)

reloaded_index = faiss.read_index(faiss_index_path)
print(f"Saved and reloaded index with {reloaded_index.ntotal} vectors at {faiss_index_path}")


## What we learned

**Terms to keep, in plain English:**

| Term | Plain meaning |
|---|---|
| Vector store | Anything that holds embeddings and can rank them by similarity |
| O(n) search | Comparing a query against every single stored item, one by one |
| Approximate nearest neighbor (ANN) | Trading a little accuracy for a lot of speed at scale |
| IndexFlatIP | FAISS's exact, no-shortcuts search using inner product |
| Vector database | A store that bundles vectors + documents + metadata + persistence |

We built a real, working vector search engine from scratch, confirmed FAISS does the identical thing faster at scale, and saw Chroma wrap the same idea in a friendlier package. "A vector database is optimized similarity search" should now feel obvious, not mysterious.

**Exercises:**

- Add a 6th story, embed it, add it to `faiss_store`, and confirm it's retrievable.
- Extend `SimpleVectorStore` with a `delete_document(filename)` method — how would you find and remove the right entry?
- Look up `faiss.IndexIVFFlat` — how is it different from `IndexFlatIP`, and why would you choose it?
